In [9]:
from torch_geometric.nn import HGTConv, Linear
from torch_geometric.loader import HGTLoader
from torch_geometric.data import HeteroData
from tqdm import tqdm
import torch.nn.functional as F
import pickle
import torch.nn as nn
import pandas as pd
from utils import *
import random
import torch
import copy

In [8]:
config = {
    "num_samples": 512,
    "batch_size": 164,
    "dropout": 0.5,
    "epochs": 50
}

In [10]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
node_drug = 'drug'
node_disease = 'disease'
rel_indication = 'indication'

In [7]:
primekg_file = '../data/kg.csv'
df = pd.read_csv(primekg_file, sep =",")
# 数据结构
# relation,display_relation,x_index,x_id,x_type,x_name,x_source,y_index,y_id,y_type,y_name,y_source

/tmp/ipykernel_40398/1999017348.py:2: DtypeWarning: Columns (3,8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(primekg_file, sep =",")


In [22]:
print(len(df))

5683172


###删除没有indication

In [11]:
# 确保每条边连接的是 drug 和 disease（顺序不限）
valid_rows = (
    ((df['x_type'] == 'drug') & (df['y_type'] == 'disease')) |
    ((df['x_type'] == 'disease') & (df['y_type'] == 'drug'))
)
drug_disease_pairs = df[(df['relation'] == 'indication') & valid_rows]

# 提取所有 x 和 y 的 (type, index) 对
x_mask = drug_disease_pairs['x_type'].isin([node_drug, node_disease])
y_mask = drug_disease_pairs['y_type'].isin([node_drug, node_disease])

# 合并所有有效实体
all_entities = pd.concat([
    drug_disease_pairs.loc[x_mask, ['x_type', 'x_index']].rename(columns={'x_type': 'type', 'x_index': 'index'}),
    drug_disease_pairs.loc[y_mask, ['y_type', 'y_index']].rename(columns={'y_type': 'type', 'y_index': 'index'})
])

# 分别提取 drug 和 disease
drugs = all_entities[all_entities['type'] == node_drug]['index'].unique().tolist()
diseases = all_entities[all_entities['type'] == node_disease]['index'].unique().tolist()

# 确保是 set 以加速
valid_drugs = set(drugs)
valid_diseases = set(diseases)

# 定义检查函数 (向量化操作的核心是避免 apply，但这里逻辑稍复杂，用布尔掩码最清晰)
# 检查 x 节点是否有效
check_x = (
    (~df['x_type'].isin(['drug', 'disease'])) |  # 情况1: 不是目标类型 -> 有效
    ((df['x_type'] == 'drug') & df['x_index'].isin(valid_drugs)) |      # 情况2: 是drug且在列表 -> 有效
    ((df['x_type'] == 'disease') & df['x_index'].isin(valid_diseases))  # 情况3: 是disease且在列表 -> 有效
)

# 检查 y 节点是否有效
check_y = (
    (~df['y_type'].isin(['drug', 'disease'])) |
    ((df['y_type'] == 'drug') & df['y_index'].isin(valid_drugs)) |
    ((df['y_type'] == 'disease') & df['y_index'].isin(valid_diseases))
)

# 只有 x 和 y 同时有效，才保留
df_cleaned = df[check_x & check_y].reset_index(drop=True)

# 1. 构建格式化后的列 (使用 f-string 或 vectorized string 操作)
# 注意：确保 index 列是字符串类型，防止数字和字符串拼接报错
head_nodes = df_cleaned['x_type'] + '::' + df_cleaned['x_index'].astype(str)
tail_nodes = df_cleaned['y_type'] + '::' + df_cleaned['y_index'].astype(str)

# 2. 组装新的 DataFrame
new_df = pd.DataFrame({
    0: head_nodes,
    1: df_cleaned['relation'],
    2: tail_nodes
})

# 3. 去重并转换为列表
# drop_duplicates() 会移除完全相同的行 (头-关系-尾 都相同)
df = new_df.drop_duplicates()
triplets = df.values.tolist()

# 打印预览
print(f"生成三元组数量: {len(triplets)}")
print(f"示例数据: {triplets[:3]}")

生成三元组数量: 5683172
示例数据: [['gene/protein::0', 'protein_protein', 'gene/protein::8889'], ['gene/protein::1', 'protein_protein', 'gene/protein::2798'], ['gene/protein::2', 'protein_protein', 'gene/protein::5646']]


In [ ]:
entity_dictionary = {}

for src, _, dest in triplets:
    for node in [src, dest]:
        n_type, n_id = node.split('::', 1)
        
        # setdefault: 如果 key 不存在，初始化为空字典，并返回该字典
        type_dict = entity_dictionary.setdefault(n_type, {})
        
        # 如果实体不在字典中，赋予新 ID (当前长度)
        if node not in type_dict:
            type_dict[node] = len(type_dict)
            


In [13]:
# 打印结果预览
for t, mapping in entity_dictionary.items():
    print(f"Type: {t}, Count: {len(mapping)}, Sample: {list(mapping.items())[:3]}")

Type: gene/protein, Count: 27573, Sample: [('gene/protein::0', 0), ('gene/protein::8889', 1), ('gene/protein::1', 2)]
Type: drug, Count: 1801, Sample: [('drug::14014', 0), ('drug::14016', 1), ('drug::14017', 2)]
Type: disease, Count: 1363, Sample: [('disease::33577', 0), ('disease::36035', 1), ('disease::38121', 2)]
Type: effect/phenotype, Count: 15082, Sample: [('effect/phenotype::24199', 0), ('effect/phenotype::25520', 1), ('effect/phenotype::23548', 2)]
Type: biological_process, Count: 28642, Sample: [('biological_process::39898', 0), ('biological_process::48388', 1), ('biological_process::39899', 2)]
Type: molecular_function, Count: 11169, Sample: [('molecular_function::53517', 0), ('molecular_function::115051', 1), ('molecular_function::115052', 2)]
Type: cellular_component, Count: 4176, Sample: [('cellular_component::55515', 0), ('cellular_component::124222', 1), ('cellular_component::124223', 2)]
Type: exposure, Count: 780, Sample: [('exposure::61677', 0), ('exposure::61678', 1)

In [14]:
from collections import defaultdict

# 使用 defaultdict 自动初始化列表，避免 if-else 判断
edge_dictionary = defaultdict(list)

for src, relation, dest in triplets:
    # 1. 解析类型 (只取 '::' 之前的部分)
    src_type = src.split('::', 1)[0]
    dest_type = dest.split('::', 1)[0]
    
    # 2. 获取整数 ID (直接从之前构建的 entity_dictionary 中查找)
    src_int_id = entity_dictionary[src_type][src]
    dest_int_id = entity_dictionary[dest_type][dest]
    
    # 3. 构建边类型键 (SrcType, Relation, DstType)
    etype = (src_type, relation, dest_type)
    
    # 4. 添加边 (defaultdict 会自动处理列表初始化)
    edge_dictionary[etype].append((src_int_id, dest_int_id))

# 如果需要转回普通字典 (可选，通常 defaultdict 也能直接用于后续处理)
edge_dictionary = dict(edge_dictionary)


In [ ]:
print(len(edge_dictionary))
# 打印预览
for etype, edges in edge_dictionary.items():
    print(f"Edge Type: {etype}, Count: {len(edges)}, Sample: {edges[:3]}")

In [ ]:
data = HeteroData()

# 1. 处理节点特征 (Node Features)
# 将字典键转为列表以便通过索引 i 进行区分
node_types = list(entity_dictionary.keys())

for i, key in enumerate(node_types):
    num_nodes = len(entity_dictionary[key])
    
    # 临时特征生成逻辑 (保持原样)
    if key == 'drug':
        # Drug: [N, 767] 随机特征
        x = torch.rand((num_nodes, 767))
    else:
        # Others: [N, 768] 填充为类型索引 i
        x = torch.ones((num_nodes, 768)) * i
    
    data[key].x = x
    data[key].id = torch.arange(num_nodes)

# 2. 处理边索引 (Edge Index)
for etype, edges in edge_dictionary.items():
    # 将 [(src, dst), ...] 列表转换为 Tensor: [N, 2] -> [2, N]
    # 原代码逻辑：先转置再 contiguous，这里用更直观的 permute 或直接构造
    edge_tensor = torch.tensor(edges, dtype=torch.long).t().contiguous()
    
    # 注意：HeteroData 的键通常是 ('src_type', 'relation', 'dst_type') 元组
    data[etype].edge_index = edge_tensor
    


In [ ]:
# 打印数据结构预览
print(data)

In [21]:

# 1. 加载 Embedding 数据
embeddings = pd.read_pickle('../data/entities_embeddings.pkl')
smiles_embeddings = pd.read_pickle('../data/smiles_embeddings.pkl')

# 获取 drug 类型的映射字典，避免重复查找
drug_mapping = entity_dictionary.get('drug', {})

# 2. 填充 Drug 节点特征
# 使用 iterrows 遍历，检查 ID 是否存在 -> 获取内部 ID -> 赋值
for _, row in smiles_embeddings.iterrows():
    ent_id = row['id']
    if ent_id in drug_mapping:
        internal_id = drug_mapping[ent_id]
        # 确保 tensor 类型一致并赋值
        data['drug'].x[internal_id] = torch.tensor(row['embedding'], dtype=torch.float32)

# 3. 填充其他类型节点特征
# 预获取所有非 drug 的节点类型集合，提高判断效率
valid_node_types = set(data.node_types) - {'drug'}

for _, row in embeddings.iterrows():
    full_id = row['id']
    # 解析类型 (split '::' 取第一部分)
    node_type = full_id.split('::', 1)[0]
    
    # 检查条件：类型有效 AND 不是 drug AND ID 存在于映射中
    if node_type in valid_node_types and full_id in entity_dictionary[node_type]:
        internal_id = entity_dictionary[node_type][full_id]
        
        # 赋值逻辑：替换前 768 维 (原代码逻辑 [:768])
        # 注意：如果 embedding 维度正好是 768，切片操作也是安全的
        data[node_type].x[internal_id][:768] = torch.tensor(row['embedding'], dtype=torch.float32)

# 验证填充结果 (可选)
print("Drug features sample:", data['drug'].x[0][:5])
print("Other features sample:", data[list(valid_node_types)[0]].x[0][:5] if valid_node_types else "No other types")

Drug features sample: tensor([-2.5625, -2.2471, -2.6396, -1.7999, -3.4816])
Other features sample: tensor([-0.1253,  0.3460,  0.3330,  0.3047,  0.4081])


In [ ]:
file = open('../data/CV data/train1.pkl', 'rb')
train_data = pickle.load(file)

file = open('../data/CV data/val1.pkl', 'rb')
val_data = pickle.load(file)

In [ ]:
# 2. 创建 Mask
# 获取边数 (逻辑与原代码完全一致)
drug_disease_num = train_data[(node_type1, rel, node_type2)]['edge_index'].shape[1]

# 随机采样 80% 的索引
mask = random.sample(range(drug_disease_num), int(drug_disease_num * 0.8))

# 初始化正向边 mask 并赋值
train_data[(node_type1, rel, node_type2)]['mask'] = torch.zeros(drug_disease_num, dtype=torch.bool)
train_data[(node_type1, rel, node_type2)]['mask'][mask] = True

# 初始化反向边 mask 并赋值 (逻辑与原代码完全一致，保持显式写出以便阅读)
train_data[(node_type2, rel, node_type1)]['mask'] = torch.zeros(drug_disease_num, dtype=torch.bool)
train_data[(node_type2, rel, node_type1)]['mask'][mask] = True